### Experiment 4
Develop an AI-powered system that automates resume screening using NLP to extract attributes like skills, experience, and qualifications. 
Apply rule-based filters for initial selection and use a neural network model trained on labeled resumes to classify candidates
into shortlisted, rejected or under review.


In [1]:
%pip install spacy

   ---------------------------------------- 0.0/14.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.2 MB 4.3 MB/s eta 0:00:04
   ----- ---------------------------------- 1.8/14.2 MB 6.1 MB/s eta 0:00:03
   --------- ------------------------------ 3.4/14.2 MB 6.6 MB/s eta 0:00:02
   ------------- -------------------------- 4.7/14.2 MB 6.6 MB/s eta 0:00:02
   ----------------- ---------------------- 6.3/14.2 MB 6.8 MB/s eta 0:00:02
   ------------------- -------------------- 6.8/14.2 MB 6.2 MB/s eta 0:00:02
   ---------------------- ----------------- 8.1/14.2 MB 6.2 MB/s eta 0:00:01
   --------------------------- ------------ 9.7/14.2 MB 6.3 MB/s eta 0:00:01
   ------------------------------ --------- 10.7/14.2 MB 6.3 MB/s eta 0:00:01
   ---------------------------------- ----- 12.3/14.2 MB 6.3 MB/s eta 0:00:01
   -------------------------------------- - 13.6/14.2 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 14.2/14.2 MB 6.3 MB/s eta 0:00:00
  

In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --- ------------------------------------ 1.0/12.8 MB 7.7 MB/s eta 0:00:02
     ------- -------------------------------- 2.4/12.8 MB 6.7 MB/s eta 0:00:02
     ---------- ----------------------------- 3.4/12.8 MB 6.7 MB/s eta 0:00:02
     --------------- ------------------------ 5.0/12.8 MB 6.6 MB/s eta 0:00:02
     ------------------- -------------------- 6.3/12.8 MB 6.6 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 6.7 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 6.8 MB/s eta 0:00:01
     --------------------------------- ------ 10.7/12.8 MB 6.8 MB/s eta 0:00:01
     ------------------------------------- -- 12.1/12.8 MB 6.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 6.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import spacy
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

In [4]:
# load the nlp pipeline
nlp = spacy.load('en_core_web_sm')

In [15]:
import random

roles = [
    "Software Engineer", "Junior Developer", "Senior Backend Engineer",
    "Data Scientist", "Receptionist", "Administrative Assistant",
    "Marketing Specialist", "IT Support", "Project Manager",
    "Graphic Designer", "Data Analyst", "Intern", "Web Developer",
    "Business Analyst", "Network Engineer", "UX Designer",
    "Content Writer", "Customer Service Representative", "QA Tester",
    "Systems Architect", "Mobile Developer", "DevOps Engineer",
    "Financial Analyst", "HR Specialist", "Operations Manager"
]

skills = [
    "Python", "Java", "C++", "JavaScript", "React", "HTML", "CSS",
    "SQL", "Excel", "WordPress", "Agile methodologies", "Adobe Creative Suite",
    "Machine Learning", "Embedded Systems", "Digital Campaigns", "APIs",
    "Troubleshooting", "Office Management", "Statistics", "Customer Service",
    "Cloud Computing", "Cybersecurity", "Data Visualization", "Project Planning"
]

degrees = [
    "Degree in CS", "Bachelor in Information Systems",
    "Master’s in Electrical Engineering", "Diploma in Web Development",
    "Bachelor’s in Statistics", "MBA in Marketing",
    "Associate Degree in IT", "PhD in Computer Science",
    "Certificate in Graphic Design", "Diploma in Business Administration",
    "Bachelor in Economics", "Master’s in Data Science",
    "Diploma in Networking", "Bachelor in Arts", "Certificate in UX Design"
]

def generate_resume():
    role = random.choice(roles)
    exp_years = random.randint(0, 15)
    skillset = random.sample(skills, 2)
    degree = random.choice(degrees)
    if exp_years == 0:
        return f"Intern with 6 months experience in {skillset[0]} and {skillset[1]}."
    else:
        return f"{role}, {exp_years} years experience in {skillset[0]} and {skillset[1]}. {degree}."

training_resumes = [generate_resume() for _ in range(500)]

# Preview first 20
for r in training_resumes[:20]:
    print(r)


Customer Service Representative, 13 years experience in JavaScript and Cybersecurity. Certificate in UX Design.
Project Manager, 3 years experience in Digital Campaigns and Embedded Systems. Associate Degree in IT.
Business Analyst, 7 years experience in CSS and Statistics. Bachelor in Information Systems.
HR Specialist, 12 years experience in SQL and Adobe Creative Suite. Bachelor’s in Statistics.
IT Support, 12 years experience in Office Management and CSS. Bachelor in Information Systems.
Software Engineer, 2 years experience in Embedded Systems and Machine Learning. Bachelor in Arts.
Financial Analyst, 14 years experience in React and Embedded Systems. Associate Degree in IT.
Web Developer, 2 years experience in Excel and Data Visualization. Diploma in Business Administration.
Network Engineer, 13 years experience in Customer Service and Project Planning. Diploma in Web Development.
Administrative Assistant, 2 years experience in C++ and Project Planning. Diploma in Business Admini

In [16]:
# Assign labels logically based on resume content
# 0 = Rejected, 1 = Under Review, 2 = Selected

import re

# Skills considered technical/high-value for a tech-focused job
tech_skills = {
    'python', 'java', 'c++', 'javascript', 'sql', 'react', 'machine learning',
    'cloud computing', 'apis', 'data visualization', 'cybersecurity', 'agile methodologies',
    'embedded systems', 'html', 'css'
}

# Degrees ranked by strength
strong_degrees = {'phd', 'master', 'mba', 'degree in cs'}
medium_degrees = {'bachelor', 'associate'}
weak_degrees = {'diploma', 'certificate'}

def assign_label(resume_text):
    text = resume_text.lower()
    
    # Extract years of experience
    years_match = re.search(r'(\d+)\s+years?\s+experience', text)
    months_match = re.search(r'6 months', text)
    
    if months_match:
        years = 0
    elif years_match:
        years = int(years_match.group(1))
    else:
        years = 0
    
    # Count technical skills present
    skill_count = sum(1 for skill in tech_skills if skill in text)
    
    # Assess degree level
    if any(d in text for d in strong_degrees):
        degree_score = 3
    elif any(d in text for d in medium_degrees):
        degree_score = 2
    elif any(d in text for d in weak_degrees):
        degree_score = 1
    else:
        degree_score = 0
    
    # Scoring logic
    score = 0
    score += min(years, 10)          # up to 10 pts for experience
    score += skill_count * 2         # 2 pts per relevant skill
    score += degree_score * 2        # up to 6 pts for degree
    
    # Classify based on score
    if score >= 16:
        return 2  # Selected
    elif score >= 8:
        return 1  # Under Review
    else:
        return 0  # Rejected

labels = [assign_label(resume) for resume in training_resumes]

In [17]:
len(labels)

500

In [18]:
# nlp extraction - flattened into a list of "Features" text
processed_profiles = []
for text in training_resumes:
    doc = nlp(text)
    entities = [ent.text.lower() for ent in doc.ents if ent.label_ in ["ORG", "PRODUCT", "DATE", "GPE"]]
    # Combine original text with extracted focus keywords
    combined = text.lower() + " " + " ".join(entities)  # set() removes duplicates
    processed_profiles.append(combined)



In [19]:
processed_profiles

['customer service representative, 13 years experience in javascript and cybersecurity. certificate in ux design. customer service representative 13 years javascript cybersecurity',
 'project manager, 3 years experience in digital campaigns and embedded systems. associate degree in it. 3 years digital campaigns and embedded systems',
 'business analyst, 7 years experience in css and statistics. bachelor in information systems. 7 years css statistics',
 'hr specialist, 12 years experience in sql and adobe creative suite. bachelor’s in statistics. 12 years sql bachelor’s statistics',
 'it support, 12 years experience in office management and css. bachelor in information systems. 12 years office management css',
 'software engineer, 2 years experience in embedded systems and machine learning. bachelor in arts. software engineer 2 years embedded systems arts',
 'financial analyst, 14 years experience in react and embedded systems. associate degree in it. 14 years react embedded systems',
 

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(processed_profiles, labels, test_size=0.35, random_state=42)

In [21]:
# Vectorization (coverting text to numbers)
vectorizer = CountVectorizer()
X_train= vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)    

In [22]:
# neural network model (multi-layer preceptron)
clf = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)

In [23]:
from sklearn.metrics import classification_report, accuracy_score

In [24]:
# Predictions
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Rejected", "Under Review", "Selected"]))


Accuracy: 0.7657142857142857
              precision    recall  f1-score   support

    Rejected       0.79      0.50      0.61        22
Under Review       0.73      0.82      0.78        84
    Selected       0.81      0.78      0.79        69

    accuracy                           0.77       175
   macro avg       0.78      0.70      0.73       175
weighted avg       0.77      0.77      0.76       175



In [25]:
new_resume = "4 years of Python experience and a Master's degree."
new_doc = nlp(new_resume)
new_entities = [ent.text.lower() for ent in new_doc.ents if ent.label_ in ["ORG", "PRODUCT", "DATE", "GPE"]]
new_features = vectorizer.transform([new_resume.lower() + " " + " ".join(new_entities)])

prediction = clf.predict(new_features)[0]
categories = {0 : "Rejected", 1 : "Under Review", 2 : "Selected"}

print(f"Candidate Resume : {new_resume}")
print(f"System Decision : {categories[prediction]}")


Candidate Resume : 4 years of Python experience and a Master's degree.
System Decision : Under Review
